# Eff-3DPSeg on Google Colab

Before running: **Runtime -> Change runtime type -> GPU (T4)**. The model uses a pure-PyTorch sparse convolution backend, so no spconv install is needed.

Pheno4D is downloaded directly from the University of Bonn (4.4 GB). No Kaggle account or token is needed. Checkpoints go to local session storage (`/content/...`) and are wiped when the runtime ends: run the last cell to download them.

In [ ]:
# ---- edit this ----
REPO_URL = 'https://github.com/gitmate21/3d-phenomics.git'
# --------------------
import os, torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> GPU, then Runtime -> Restart session.')

# Local session storage: fast, but wiped when the Colab runtime is recycled.
# Download the checkpoint (last cell) before the session ends.
CKPT_DIR = '/content/checkpoints'
LOG_DIR  = '/content/logs'
PRED_DIR = '/content/preds'
for d in (CKPT_DIR, LOG_DIR, PRED_DIR):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Clone, or update an existing clone, then enter the project folder.
REPO = '/content/3d-phenomics'
if os.path.isdir(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone {REPO_URL} {REPO}
%cd {REPO}/eff3dpseg

In [ ]:
# Sparse-conv backend. spconv has no working build for Colab's CUDA 13 (its C++ build fails),
# so use the pure-PyTorch implementation in scripts/sparse_torch.py: no install, any CUDA, a bit slower.
# Later cells run `!python scripts/...` which inherits this environment variable.
os.environ['EFF_SPARSE_BACKEND'] = 'torch'     # 'auto' tries spconv first, then falls back
!python scripts/test_sparse_torch.py

In [ ]:
# Download Pheno4D directly from the University of Bonn (4.4 GB zip; several minutes).
# Check the dataset's terms on https://www.ipb.uni-bonn.de/data/pheno4d/ before using or sharing it.
PHENO4D_URL = 'https://www.ipb.uni-bonn.de/html/projects/Pheno4D/Pheno4D.zip'
RAW_ROOT = '/content/pheno4d_raw'
if not os.path.isdir(RAW_ROOT):
    !wget -q --show-progress -O /content/Pheno4D.zip {PHENO4D_URL}
    !unzip -q /content/Pheno4D.zip -d {RAW_ROOT}
    !rm /content/Pheno4D.zip          # free ~4.4 GB of disk
!ls {RAW_ROOT}

In [ ]:
# RAW_DIR must be the folder that directly contains the plant folders (Maize01, Maize02, ...).
# The zip's internal layout is not assumed: find where Maize01 lives.
import glob
hits = glob.glob(f'{RAW_ROOT}/**/Maize01', recursive=True)
assert hits, f'Maize01 not found under {RAW_ROOT}; look at the listing above'
RAW_DIR = os.path.dirname(hits[0])
print('RAW_DIR =', RAW_DIR)
!ls {RAW_DIR}
# Check the file format: expect x y z then a label column. prepare_pheno4d.py assumes label = column 3.
!ls {RAW_DIR}/Maize01 | head -5
!head -n 3 {RAW_DIR}/Maize01/$(ls {RAW_DIR}/Maize01 | grep _a.txt | head -1)

In [ ]:
!python scripts/smoke_test.py

In [ ]:
# Convert raw txt -> npz + plant-level splits (whole plants are held out, so no leakage).
# Reading ~50 MB text files is slow (several minutes for the whole set); this runs once.
SPECIES = 'Tomato'                        # 'Tomato' (Tomato01-07) or 'Maize' (Maize01-07); one species per run
DATA = f'/content/pheno4d_npz_{SPECIES.lower()}'
plants = sorted(d for d in os.listdir(RAW_DIR) if os.path.isdir(f'{RAW_DIR}/{d}') and d.startswith(SPECIES))
print('plants:', plants)
VAL, TEST = plants[-2], plants[-1]        # edit to hold out other plants
print('val:', VAL, '| test:', TEST)
!python scripts/prepare_pheno4d.py --raw {RAW_DIR} --out {DATA} --plants {' '.join(plants)} --val-plants {VAL} --test-plants {TEST}

In [ ]:
# ---------------- run settings (printed and saved with every run as <run>_params.json) ----------------
VOXEL_SIZE = 0.2                  # <-- SET from the printed extent (plant ~50-200 voxels tall; the paper used 0.5 for tomato)
LABELED_POINTS = [50, 100, 200]   # one training run per value (the paper's settings). 0 = all points labeled
DROP_SOIL = True                  # True: delete soil points as the paper does. False: keep them as input
MAX_ITER, BATCH_SIZE = 2000, 1    # the paper used 4000 iterations with batch size 2
HOURS_PER_RUN = 3.3               # each run saves and stops at this limit; --resume auto continues it later
# Quick trial first: MAX_ITER = 20 and LABELED_POINTS = [50] shows the run report, speed and memory in a few minutes.
# Checkpoints live in /content (wiped when the runtime ends): run the last cell to download them.
# --------------------------------------------------------------------------------------------------------
SOIL = '--drop-soil' if DROP_SOIL else '--keep-soil'
RUNS = [f"{SPECIES.lower()}_{'full' if k == 0 else f'w{k}'}" for k in LABELED_POINTS]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for k, run in zip(LABELED_POINTS, RUNS):
    print(f"\n################ {run}: {k or 'all'} labeled points ################")
    !python scripts/train.py --data {DATA} --run-name {run} --labeled-points {k} {SOIL} \
        --max-iter {MAX_ITER} --batch-size {BATCH_SIZE} --voxel-size {VOXEL_SIZE} --validate-step 50 --save-every 50 \
        --ckpt-dir {CKPT_DIR} --log-dir {LOG_DIR} --resume auto --time-limit-hours {HOURS_PER_RUN}

In [ ]:
# Evaluate every run's best checkpoint on the validation and test plants (all raw points, soil excluded) and print
# precision / recall / F1 / IoU next to the matching paper rows (Table 3, tomato). Voxel size and soil handling are
# read from each checkpoint. Use --which latest to evaluate the final iteration instead of the best one.
!python scripts/compare_runs.py --data {DATA} --ckpt-dir {CKPT_DIR} --runs {' '.join(RUNS)} --phases val test

# Local storage is wiped when the runtime ends: download checkpoints, logs and the run-parameter files first.
!zip -qr /content/results.zip {CKPT_DIR} {LOG_DIR}
from google.colab import files
files.download('/content/results.zip')